# Air Quality Data Analysis with Python
## Notebook 8 · Separating Regional Background from Local Sources

⏱️ About 80 minutes &nbsp;·&nbsp; ⬅️ Builds on Notebooks 5 and 7 &nbsp;·&nbsp; 🔬 The advanced one

Here is the question every city eventually asks: **if we cleaned up our own
emissions, how much cleaner would the air actually get?**

It is not rhetorical. A city that is 30 µg/m³ because of its own traffic can fix
that with its own policy. A city that is 30 µg/m³ because Saharan dust is
blowing over it cannot — and a mayor who spends a transport budget trying will
have nothing to show for it.

So you need to split each measurement into:

* a **regional background** — the load that arrives over the whole city from
  outside, which local policy cannot touch, and
* a **local contribution** — what this particular neighbourhood adds on top,
  which local policy can.

Notebook 7 left you with a puzzle. Between February and August, Accra got much
**cleaner** (about 32 → 22 µg/m³) and at the same time much more **unequal**
(spread 23 → 33 µg/m³). One site, Agbogbloshie, didn't drop at all.

The method of **Zimmerman et al. (2020)** explains all of it at once. It's a
wavelet decomposition that separates pollution by *how fast it changes*, then
uses the shape of the whole network to find the background.

You'll learn:

* why the speed of a signal tells you the size of its source,
* what a wavelet decomposition actually does (it is less mysterious than it sounds),
* the iterative baseline trick that keeps every component positive,
* the min-across-sites rule that defines the regional background,
* and how to read the answer off a diurnal profile.

### 1. Setup

One new library: **PyWavelets** (`pywt`). Colab already has it; the cell below
installs it only if it's missing.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import pywt
except ImportError:  # not on this machine — fetch it once
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
    import pywt

DATA = "../data"  # local checkout of the course repository
if not Path(DATA).exists():  # running in Colab -> read from GitHub
    DATA = "https://raw.githubusercontent.com/rwpinder/tutorial-air-quality-data-analysis/main/data"

GRAY, BLUE, ORANGE, GREEN = "#999999", "#0072B2", "#D55E00", "#009E73"
plt.rcParams.update({
    "figure.figsize": (9.5, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": "#cbcbcb", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": "#333333", "xtick.color": "#333333", "ytick.color": "#333333",
    "legend.frameon": False,
})


def check(name, test, hint=""):
    """Run test() and print a friendly ✅ or 💡 — never an error message."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    if ok:
        print(f"✅ {name} — looks right!")
    else:
        print(f"💡 {name} — not quite yet. Hint: {hint}")


def load_month(filename, start, end):
    """Read a network file and lay it out as one gap-free hourly column per site."""
    df = pd.read_csv(f"{DATA}/{filename}")
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
    wide = df.pivot_table(index="datetime", columns="site_name", values="pm25_value")
    hours = pd.date_range(start, end, freq="h", tz="UTC")
    return wide.reindex(hours)


feb = load_month("accra_network_feb2025.csv", "2025-02-01", "2025-02-28 23:00")
aug = load_month("accra_network_aug2025.csv", "2025-08-01", "2025-08-31 23:00")

print(f"February 2025: {feb.shape[1]} sites x {feb.shape[0]} hours")
print(f"August 2025:   {aug.shape[1]} sites x {aug.shape[0]} hours")
print(f"\nFebruary sites: {', '.join(feb.columns)}")

### 2. Why speed tells you size

The central idea is physical, not mathematical.

Think about what a plume has to do to reach your sensor. A **cooking fire 50 m
upwind** appears and disappears in minutes — the wind shifts and it's gone. A
**congested junction 2 km away** builds through the morning rush and fades by
mid-morning: hours, not minutes. A **dust layer blown 800 km from the Sahel**
sits over the entire city for days.

Distance and duration go together. Something far away has to be big and
long-lived to reach you at all; something near you can flicker.

So Zimmerman et al. split the signal by how fast it varies:

| Component | Changes over | Physically |
|---|---|---|
| **Neighbourhood** | faster than ~2 h | the street you're standing on |
| **Urban** | ~2–8 h | the city's own daily rhythm — traffic, cooking |
| **Regional** | slower than ~8 h | air masses arriving from far away |

Splitting a signal by speed is exactly what a **wavelet decomposition** does.

### 3. What a wavelet decomposition actually does

Take one sensor for a week. `pywt.wavedec` breaks the series into a stack of
layers: a smooth **approximation** plus a set of **details**, each detail
holding the wiggles at one timescale.

The useful move is to keep the approximation, throw the details away, and
rebuild. What comes back is the original with everything fast erased — a
baseline.

In [ ]:
site = "Kaneshie Market"
# .dropna() matters: a wavelet transform has no notion of "missing", and a
# single NaN spreads through every coefficient and out into every component.
week = feb[site].loc["2025-02-08":"2025-02-14"].interpolate(limit=6).dropna()
# np.array(..., dtype=float) rather than .values: pandas hands back a read-only
# view, and pywt writes into the array it is given.
week_values = np.array(week, dtype=float)

print(f"{len(week)} usable hours in the sample week")

coeffs = pywt.wavedec(week_values, "db4", level=3)
print(f"{len(week)} hours -> {len(coeffs)} coefficient arrays")
print(f"  approximation : {len(coeffs[0])} coefficients (the slow part)")
for i, detail in enumerate(coeffs[1:], start=1):
    print(f"  detail {i}      : {len(detail)} coefficients")

# rebuild using the approximation alone: details zeroed out
smooth_only = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
baseline = pywt.waverec(smooth_only, "db4")[:len(week_values)]

fig, ax = plt.subplots()
ax.plot(week.index, week_values, color=GRAY, linewidth=1.0, label="measured")
ax.plot(week.index, baseline, color=BLUE, linewidth=2.5, label="wavelet baseline")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(loc="upper right")
ax.set_title(f"{site}, one Harmattan week: the baseline keeps the slow story")
plt.show()

`level=3` with hourly data means "smooth away anything faster than about
2³ = 8 hours". The blue line follows the multi-day swell and ignores the daily
spikes.

Now look closely at the blue line where the grey line dips. **The baseline goes
above the measurement.** If we call the fast component `measured - baseline`,
it comes out negative — and a source that contributes *negative* PM2.5 is not a
thing. Every component has to be ≥ 0 or the decomposition can't be read as
"how much came from where".

In [ ]:
naive_fast = week_values - baseline
print(f"hours where the naive fast component is negative: "
      f"{(naive_fast < 0).sum()} of {len(naive_fast)}")
print(f"most negative value: {naive_fast.min():.1f} µg/m³")

### 4. The iterative baseline

The fix, from Klems et al. (2010) and used by Zimmerman et al., is stubbornly
simple: **wherever the baseline rose above the data, pull it back down to the
data, then re-smooth. Repeat until it stops moving.**

The result is a baseline that never exceeds the measurement, so the component
above it is never negative.

This function is given to you — the logic is fiddly and the point of the
notebook is what you do with it, not typing it out.

In [ ]:
def iterative_baseline(values, wavelet="db4", level=3, max_iterations=100, tolerance=0.01):
    """A non-negative baseline: wavelet-smooth, clip to the data, repeat.

    Follows Klems et al. (2010) as implemented by Zimmerman et al. (2020).
    Guarantees baseline <= values, so `values - baseline` is never negative.
    """
    n = len(values)
    current = values.copy()
    baseline = current
    for _ in range(max_iterations):
        coeffs = pywt.wavedec(current, wavelet, level=level)
        approx = pywt.waverec([coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]],
                              wavelet)[:n]
        # where the smooth curve sits above the data, adopt the data instead
        baseline = np.where(current - approx < 0, current, approx)
        with np.errstate(divide="ignore", invalid="ignore"):
            change = np.where(approx != 0, (baseline - approx) / np.abs(approx), 0)
        if np.nanmean(np.abs(change)) < tolerance:
            break
        current = baseline
    baseline = np.where(baseline > values, values, baseline)
    baseline = np.where(baseline < 0, values, baseline)
    return baseline


fixed = iterative_baseline(week_values, level=3)
print(f"hours where the fast component is negative: "
      f"{((week_values - fixed) < 0).sum()} of {len(fixed)}")

fig, ax = plt.subplots()
ax.plot(week.index, week_values, color=GRAY, linewidth=1.0, label="measured")
ax.plot(week.index, baseline, color=ORANGE, linewidth=1.6, linestyle="--",
        label="plain wavelet baseline")
ax.plot(week.index, fixed, color=BLUE, linewidth=2.5, label="iterative baseline")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(loc="upper right")
ax.set_title("The iterative baseline tucks under the measurements instead of crossing them")
plt.show()

### 5. Three components from two baselines

With a baseline generator that behaves, the three-way split is arithmetic. Build
two baselines — one that smooths away everything faster than 2 hours, one that
smooths away everything faster than 8 hours — and difference them:

* **regional** = the 8-hour baseline
* **urban** = 2-hour baseline − 8-hour baseline
* **neighbourhood** = measured − 2-hour baseline

The three add back up to very nearly the measurement, and none of them can be
negative — the two properties that let you read them as "how much came from
where". (Nearly, not exactly: see the note after the chart.)

With hourly data, "2 hours" is `level = log2(2/1) = 1` and "8 hours" is
`level = log2(8/1) = 3`. One more detail from the published method: a short
rolling **median** first, which removes single-hour spikes that would otherwise
leak across every timescale.

In [ ]:
def decompose(series, wavelet="db4"):
    """Split one hourly series into (neighbourhood, urban, regional), all >= 0."""
    values = np.array(series, dtype=float)
    smoothed = pd.Series(values).rolling(3, center=True, min_periods=1).median().values

    regional = iterative_baseline(smoothed, wavelet, level=3)   # slower than ~8 h
    two_hour = iterative_baseline(smoothed, wavelet, level=1)   # slower than ~2 h
    two_hour = np.maximum(two_hour, regional)  # the 2 h baseline can't dip below the 8 h one

    neighbourhood = np.maximum(values - two_hour, 0)
    urban = two_hour - regional
    return neighbourhood, urban, regional


nbh, urb, reg = decompose(week)
print(f"mean neighbourhood : {nbh.mean():5.1f} µg/m³")
print(f"mean urban         : {urb.mean():5.1f} µg/m³")
print(f"mean regional      : {reg.mean():5.1f} µg/m³")
print(f"sum of components  : {(nbh + urb + reg).mean():5.1f} µg/m³")
print(f"measured           : {week_values.mean():5.1f} µg/m³")

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.stackplot(week.index, reg, urb, nbh,
             labels=["Regional (> 8 h)", "Urban (2–8 h)", "Neighbourhood (< 2 h)"],
             colors=[BLUE, ORANGE, GREEN], alpha=0.9)
ax.plot(week.index, week_values, color="#333333", linewidth=0.9, label="measured")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_ylim(0, None)
ax.legend(loc="upper right", ncol=2)
ax.set_title(f"{site}, 8–14 February: the slow regional layer carries most of the load")
plt.show()

The components recover the measurement to within a percent or two here — close,
but deliberately *not* exact, and the gap is worth understanding rather than
ignoring. Two steps in `decompose` cost that accuracy on purpose:

* the baselines are built from a **median-smoothed** copy of the series, so a
  one-hour spike doesn't leak into every timescale at once, and
* the neighbourhood component is **clipped at zero**, which nudges the total up
  wherever the measurement dipped below its own 2-hour baseline.

Across all 37 site-months in this course the sum runs 0.5–4.2% above the
measurement, with a median near 1%. A noisier series dips below its baseline
more often, so it gets clipped more often — which means this small bookkeeping
error doubles as a rough data-quality signal.

So: a very good accounting of the measurement, not an algebraic identity. Quote
the components as shares, not to three decimal places.

### 6. The Zimmerman step: what is *background*?

The regional component above is still not the background. It is everything slow
at *this* sensor — and a site next to a scrapyard that smoulders for weeks has a
slow local signal too.

Zimmerman et al.'s insight (their Equation 3) is that the background is
something you can only see with a **network**. At any hour, look across every
site and take the **smallest** regional value:

$$\text{background}(t) = \min_i \; \text{regional}_i(t)$$

The reasoning: the cleanest site in the city at that moment still cannot get
below what is blowing over all of it. Whatever is left at each site is that
site's own persistent local source (their Equation 5):

$$\text{persistent local}_i(t) = \text{regional}_i(t) - \text{background}(t)$$

So each measurement ends up in four buckets — one shared by the whole city and
three belonging to the site.

In [ ]:
def decompose_network(wide, min_sites=3):
    """Decompose every site, then split regional into background + persistent local."""
    parts = {}
    for name in wide.columns:
        series = wide[name].interpolate(limit=6).dropna()
        if len(series) < 32:  # too short for a level-3 decomposition
            continue
        nbh, urb, reg = decompose(series)
        parts[name] = dict(
            neighbourhood=pd.Series(nbh, index=series.index),
            urban=pd.Series(urb, index=series.index),
            regional=pd.Series(reg, index=series.index),
            measured=series,
        )

    regional = pd.DataFrame({k: v["regional"] for k, v in parts.items()})
    # Zimmerman Eq. 3, with two guards: ignore zeros (an offline sensor would
    # otherwise drag the background to nothing) and require a few sites to be
    # reporting before trusting the minimum at all.
    positive = regional.where(regional > 0)
    background = positive.min(axis=1)
    background[positive.notna().sum(axis=1) < min_sites] = np.nan

    for name, part in parts.items():
        aligned = background.reindex(part["regional"].index)
        part["background"] = aligned
        part["persistent_local"] = part["regional"] - aligned  # Eq. 5
    return parts, background


feb_parts, feb_background = decompose_network(feb)
aug_parts, aug_background = decompose_network(aug)

print(f"February: decomposed {len(feb_parts)} sites, "
      f"background defined for {feb_background.notna().sum()} of {len(feb_background)} hours")
print(f"August:   decomposed {len(aug_parts)} sites, "
      f"background defined for {aug_background.notna().sum()} of {len(aug_background)} hours")
print(f"\nmean regional background, February : {feb_background.mean():5.1f} µg/m³")
print(f"mean regional background, August   : {aug_background.mean():5.1f} µg/m³")

There it is, in two numbers: the air arriving over Accra carried about
**16 µg/m³** in February and about **9 µg/m³** in August. Every site in the city
was paying that toll before a single local vehicle started up.

One reassurance before we lean on that minimum. If a single unlucky sensor set
the floor every hour, the whole method would rest on that one instrument. It
doesn't — the job passes around the network:

In [ ]:
feb_regional = pd.DataFrame({k: v["regional"] for k, v in feb_parts.items()})
defines_floor = feb_regional.where(feb_regional > 0).idxmin(axis=1).value_counts()
share = (100 * defines_floor / defines_floor.sum()).round(1)
print("Share of February hours in which each site defined the citywide minimum:")
print(share.head(8).to_string())

No site sets the floor more than about a quarter of the time. The background is
a property of the network, not of its unluckiest member.

### 7. The diurnal profile of a source, not of a sensor

Notebook 5 built diurnal profiles of *measurements*. Now you can build one per
**component** — and that is a far more useful picture, because the components
have different daily shapes and the shapes say what they are.

Pick a sensor. Change `SENSOR` to any name printed below and re-run this section.

In [ ]:
print("February sites:", ", ".join(sorted(feb_parts)))
print("August sites:  ", ", ".join(sorted(aug_parts)))
print("in both:       ", ", ".join(sorted(set(feb_parts) & set(aug_parts))))

SENSOR = "Kaneshie Market"


def diurnal_components(parts, name):
    """Average each component by hour of day, on Accra local time."""
    part = parts[name]
    frame = pd.DataFrame({
        "Regional background": part["background"],
        "Persistent local": part["persistent_local"],
        "Urban (2–8 h)": part["urban"],
        "Neighbourhood (< 2 h)": part["neighbourhood"],
    }).dropna()
    local_hours = frame.index.tz_convert("Africa/Accra").hour
    return frame.groupby(local_hours).mean()


feb_profile = diurnal_components(feb_parts, SENSOR)
print(f"\n{SENSOR}, February — mean by component (µg/m³):")
print(feb_profile.mean().round(1).to_string())

Now the picture the whole notebook has been building towards: the same sensor,
the same chart, in both months.

In [ ]:
LAYERS = ["Regional background", "Persistent local", "Urban (2–8 h)", "Neighbourhood (< 2 h)"]
COLORS = [BLUE, ORANGE, GREEN, GRAY]


def plot_profile(ax, profile, title, ymax):
    ax.stackplot(profile.index, *[profile[layer] for layer in LAYERS],
                 labels=LAYERS, colors=COLORS, alpha=0.9)
    ax.set_xlim(0, 23)
    ax.set_ylim(0, ymax)
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel("Hour of day (Africa/Accra)")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title(title)


aug_profile = diurnal_components(aug_parts, SENSOR)
ymax = 1.05 * max(feb_profile.sum(axis=1).max(), aug_profile.sum(axis=1).max())

fig, (ax_feb, ax_aug) = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
plot_profile(ax_feb, feb_profile,
             f"February 2025 — background {feb_profile['Regional background'].mean():.0f} µg/m³", ymax)
plot_profile(ax_aug, aug_profile,
             f"August 2025 — background {aug_profile['Regional background'].mean():.0f} µg/m³", ymax)
ax_aug.set_ylabel("")
handles, labels = ax_feb.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f"{SENSOR}: the inherited layer roughly halves between the seasons",
             fontsize=13, fontweight="bold", x=0.015, ha="left", y=1.04)
plt.show()

Read the blue layer first. In February it is a thick slab that barely moves
across the day — because a regional dust load does not care what time it is.
That flatness is the signature of a distant source.

In August the blue slab is roughly half as thick, and what sits above it has
kept its **shape**: the morning and evening bulges of traffic and cooking from
Notebook 5. Local sources have a clock; the background does not.

### 8. The seasonal comparison, in numbers

The chart makes the case; a table settles it. For every site measured in both
months, what share of the load was regional background?

In [ ]:
def component_summary(parts, name):
    part = parts[name]
    frame = pd.DataFrame({
        "background": part["background"],
        "persistent_local": part["persistent_local"],
        "urban": part["urban"],
        "neighbourhood": part["neighbourhood"],
        "measured": part["measured"],
    }).dropna()
    means = frame.mean()
    return dict(total=means["measured"], background=means["background"],
                background_pct=100 * means["background"] / means["measured"],
                local=means["persistent_local"] + means["urban"] + means["neighbourhood"])


shared = sorted(set(feb_parts) & set(aug_parts))
rows = []
for name in shared:
    f, a = component_summary(feb_parts, name), component_summary(aug_parts, name)
    rows.append(dict(site=name,
                     feb_total=f["total"], feb_bg=f["background"], feb_bg_pct=f["background_pct"],
                     aug_total=a["total"], aug_bg=a["background"], aug_bg_pct=a["background_pct"]))
comparison = pd.DataFrame(rows).set_index("site")
print(comparison.round(1).to_string())
print()
print(f"Across the {len(shared)} sites measured in both months:")
print(f"  February — total {comparison['feb_total'].mean():.1f} µg/m³, "
      f"background {comparison['feb_bg'].mean():.1f} ({comparison['feb_bg_pct'].mean():.0f}%)")
print(f"  August   — total {comparison['aug_total'].mean():.1f} µg/m³, "
      f"background {comparison['aug_bg'].mean():.1f} ({comparison['aug_bg_pct'].mean():.0f}%)")

Notice that the `feb_bg` column is nearly the same number at every site, and so
is `aug_bg`. That is not a bug — it is the definition. The background is one
citywide series, shared by everyone. What differs between sites is how much of
their total it accounts for, and that varies enormously: from about a third to
nearly two thirds.

And here is the answer to Notebook 7's puzzle. Between February and August the
**background** falls by roughly 43%, while each site's **local** contribution
falls much less. So every site's total drops — but the sites that were *mostly*
background drop the most, and the sites that were *mostly* local barely drop at
all.

That is exactly why August is both cleaner and more unequal. The shared part of
the pollution — the part that was making every neighbourhood look alike — is the
part that went away.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))
x = np.arange(len(shared))
width = 0.38
ax.bar(x - width / 2, comparison["feb_bg"], width, color=BLUE, label="February · regional background")
ax.bar(x - width / 2, comparison["feb_total"] - comparison["feb_bg"], width,
       bottom=comparison["feb_bg"], color=BLUE, alpha=0.35, label="February · local")
ax.bar(x + width / 2, comparison["aug_bg"], width, color=ORANGE, label="August · regional background")
ax.bar(x + width / 2, comparison["aug_total"] - comparison["aug_bg"], width,
       bottom=comparison["aug_bg"], color=ORANGE, alpha=0.35, label="August · local")
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=35, ha="right")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_ylim(0, 1.35 * comparison[["feb_total", "aug_total"]].to_numpy().max())
ax.legend(ncol=2, loc="upper left")
ax.set_title("Solid = inherited from outside the city, pale = made inside it")
plt.show()

Agbogbloshie is the bar to look at. Its solid blocks are the same height as
everyone else's — it receives exactly the same regional background as the rest
of Accra. Everything else about it is pale: locally made, and almost unchanged
between the seasons.

### ✏️ Your turn 1: the local contribution

For `SENSOR` in **August**, compute the mean **local contribution** — everything
that is not regional background. Use the `aug_parts[SENSOR]` dictionary, which
has keys `persistent_local`, `urban` and `neighbourhood`.

Store the answer in `local_aug`.

In [ ]:
# each of these is a pandas Series; .mean() skips the NaN hours for you
part = aug_parts[SENSOR]
local_aug = ...

print(f"{SENSOR}, August — local contribution: {local_aug} µg/m³")

In [ ]:
check("local_aug",
      lambda: abs(local_aug - component_summary(aug_parts, SENSOR)["local"]) < 1.0,
      "add the means of persistent_local, urban and neighbourhood")

### ✏️ Your turn 2: which site is most its own problem?

A site whose background share is *low* is a site where local action pays off
most. Using the `comparison` table, find the site with the **lowest**
`feb_bg_pct` — the site that made the largest share of its own February
pollution, rather than inheriting it.

Store the site's name in `most_local`.

In [ ]:
# comparison is indexed by site name, so the answer is an index label
# — pandas has a method that returns the label of a column's smallest value
most_local = ...

print("Most locally-driven site in February:", most_local)

In [ ]:
check("most_local", lambda: most_local == "Agbogbloshie",
      "which site's feb_bg_pct is smallest? .idxmin() gives you its label")

Agbogbloshie, of course — the site that refused to get cleaner in August. Only
about a third of its February load was inherited, falling to under a fifth in
August. It is the one place in this network where local action would do almost
all of the work.

### 9. What this method can and cannot tell you

Every method has a boundary, and quoting numbers past it is how good analysis
goes wrong. Four honest limits on what you just computed:

1. **The background is a floor, not a measurement.** Taking the minimum across
   sites assumes at least one site, at each hour, has no local sources of its
   own. If every site is polluted, the true background is *lower* than this
   estimate, and the local contribution correspondingly larger.
2. **It needs a real network.** With one or two sensors there is no minimum
   worth taking. The code above refuses to define a background at any hour when
   fewer than three sites are reporting.
3. **Timescale is a proxy for distance, not a measurement of it.** A slow local
   source — a scrapyard fire that smoulders for a week — is counted as regional,
   and Agbogbloshie is exactly the kind of site where that could happen. The
   method separates *fast from slow*, and we interpret that as *near from far*.
   Usually fair; not always.
4. **One broken sensor is still in this data.** Notebook 7 found that Osu Presby
   School's August series has almost no hour-to-hour memory. It was left in so
   you could find it. It appears only in the August set, where it defines the
   citywide floor in under a tenth of the hours; removing it moves the August
   background by less than 3%. That is the sort of thing to measure rather than
   assume — the floor-share table above is how you would check it.

Point 4 is the habit worth keeping. Find the weakest part of your data, work out
which number it could contaminate, and go and check how much it actually did.

### 10. What you built

You took a single number — a PM2.5 measurement — and split it into physically
meaningful pieces using nothing but the speed at which it changed and the shape
of the network around it.

Across the 15 sites measured in both months:

| | February 2025 | August 2025 |
|---|---|---|
| Mean PM2.5 | 31.7 µg/m³ | 22.8 µg/m³ |
| Regional background | 15.6 µg/m³ (**51%**) | 8.9 µg/m³ (**43%**) |
| Locally made | 16.1 µg/m³ | 13.9 µg/m³ |
| Site spread (Notebook 7) | ~23 µg/m³ | ~33 µg/m³ |

The two notebooks now tell one story. The Harmattan delivers a thick, uniform
regional layer that raises every neighbourhood together and makes the city look
homogeneous. When it lifts, that shared layer roughly halves — and what remains
is each district's own pollution, which differs threefold across Accra. Cleaner
air and sharper inequality, from the same cause.

**Where to take this next**

* Run the decomposition on a different `SENSOR` — try `Agbogbloshie` against
  `Ashaley Botwe` and watch the blue layer go from a third of the total to
  nearly two thirds.
* Apply it to a month between the two seasons; the transition is gradual.
* Read the source: Zimmerman et al. (2020), *Aerosol and Air Quality Research*
  20, 314–328, which introduced this four-component split, and Klems et al.
  (2010) for the iterative baseline.

You have now done, by hand, what the AQ agent's wavelet module does inside its
source-attribution pipeline — on the same city, with the same method.